# Position Density Animation

Load and animate the persisted position-density snapshots for one observation session.

In [ ]:
# Load the shared project-path, database, and report-header helpers.
%run pathutils.ipynb
%run database.ipynb
%run report-header.ipynb

In [ ]:
# Select the observation session and the delay between animation frames.
session_id = 12
frame_interval_ms = 500

In [ ]:
# Display the standard database and report-generation metadata.
report_metadata = display_report_header('Position Density Animation')

In [ ]:
# Load snapshot metadata and populated cells using the shared SQL-query convention.
if not isinstance(session_id, int) or session_id <= 0:
    raise ValueError('session_id must be a positive integer')

snapshot_query = construct_query(
    'tracker',
    'reports',
    'position-density-snapshots.sql',
    {'session_id': session_id})
snapshot_cells = query_data('tracker', snapshot_query)
snapshot_cells['Captured At UTC'] = pd.to_datetime(snapshot_cells['Captured At UTC'], utc=True)
snapshot_cells.head(20)

In [ ]:
# Present one row per available frame before constructing the animation.
snapshot_summary = (snapshot_cells[[
    'Snapshot Id', 'Session Id', 'Captured At UTC', 'Position Count', 'Maximum Bin Count'
]]
.drop_duplicates('Snapshot Id')
.sort_values(['Captured At UTC', 'Snapshot Id'])
.reset_index(drop=True))
snapshot_summary

In [ ]:
# Render populated density cells as an inline animation with stable session axes and colour scaling.
import matplotlib.pyplot as plt
from matplotlib import animation, colors
from IPython.display import HTML

frame_metadata = snapshot_summary.to_dict('records')
minimum_latitude = snapshot_cells['Minimum Latitude'].min()
maximum_latitude = snapshot_cells['Maximum Latitude'].max()
minimum_longitude = snapshot_cells['Minimum Longitude'].min()
maximum_longitude = snapshot_cells['Maximum Longitude'].max()
maximum_count = max(1, int(snapshot_cells['Maximum Bin Count'].max()))
colour_scale = colors.LogNorm(vmin=1, vmax=max(2, maximum_count))
colour_map = plt.get_cmap('viridis')

def padded_limits(minimum, maximum):
    # Preserve useful axes even for legacy snapshots whose bounds collapse to one coordinate.
    padding = max((maximum - minimum) * 0.02, 0.01)
    return minimum - padding, maximum + padding

figure, axis = plt.subplots(figsize=(10, 8))
axis.set_xlim(*padded_limits(minimum_longitude, maximum_longitude))
axis.set_ylim(*padded_limits(minimum_latitude, maximum_latitude))
axis.set_xlabel('Longitude')
axis.set_ylabel('Latitude')
axis.grid(alpha=0.2)
scatter = axis.scatter(
    [float('nan')], [float('nan')], c=[1], s=[0],
    marker='h', cmap=colour_map, norm=colour_scale, edgecolors='none')
colour_bar = figure.colorbar(scatter, ax=axis, label='Position count')

def draw_frame(frame_number):
    # Select the complete persisted cell collection belonging to this snapshot.
    metadata = frame_metadata[frame_number]
    cells = snapshot_cells[
        snapshot_cells['Snapshot Id'] == metadata['Snapshot Id']
    ].dropna(subset=['Cell Latitude', 'Cell Longitude', 'Cell Count'])

    offsets = cells[['Cell Longitude', 'Cell Latitude']].to_numpy()
    counts = cells['Cell Count'].to_numpy()
    scatter.set_offsets(offsets if len(offsets) else [[float('nan'), float('nan')]])
    scatter.set_array(counts if len(counts) else [float('nan')])
    scatter.set_facecolors(colour_map(colour_scale(counts)) if len(counts) else [[0, 0, 0, 0]])
    scatter.set_sizes(30 + (counts * 18) if len(counts) else [0])
    captured = metadata['Captured At UTC'].strftime('%Y-%m-%d %H:%M:%S UTC')
    axis.set_title(
        f"Session {session_id} · Snapshot {frame_number + 1}/{len(frame_metadata)}\n"
        f"{captured} · {metadata['Position Count']:,} positions")
    return scatter,

position_density_animation = animation.FuncAnimation(
    figure,
    draw_frame,
    frames=len(frame_metadata),
    interval=frame_interval_ms,
    repeat=True,
    blit=False)
plt.close(figure)
HTML(position_density_animation.to_jshtml())